# Day 093 Solution — Sentiment Pipeline Dashboard

In [ ]:
import re

# Gate-safe mock LLM — keyword-based, deterministic, no Ollama required
# Checks only the user message to avoid matching keywords in the system prompt.
def _mock_llm(messages):
    user_text = next(
        (m.get("content", "") for m in messages if m.get("role") == "user"), ""
    ).lower()
    if any(w in user_text for w in ["surge", "rally", "rise", "gain", "bull", "strong"]):
        return "0.75"
    if any(w in user_text for w in ["crash", "fall", "decline", "bear", "weak", "loss"]):
        return "-0.60"
    return "0.10"

BULLISH_HEADLINES = [
    "Tech stocks rally on strong earnings",
    "Markets surge as Fed signals rate pause",
    "S&P 500 gains 2% on positive jobs data",
    "Bull market continues with broad gains",
]
BEARISH_HEADLINES = [
    "Markets crash amid recession fears",
    "Stocks fall sharply on weak economic data",
    "S&P 500 declines on hawkish Fed remarks",
    "Bear market deepens as losses mount",
]
NEUTRAL_HEADLINES = [
    "Markets trade sideways in quiet session",
    "Mixed signals leave investors cautious",
    "Stocks finish flat as investors await data",
]
def parse_score(text):
    """Extract and clamp a float from LLM output. Returns 0.0 if not found."""
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if not matches:
        return 0.0
    return max(-1.0, min(1.0, float(matches[0])))

def build_sentiment_prompt(headline):
    return [
        {
            "role": "system",
            "content": (
                "You are a financial news sentiment analyzer. "
                "Score the sentiment from -1.0 (very bearish) to 1.0 (very bullish). "
                "Reply with ONLY a single decimal number. No explanation."
            ),
        },
        {"role": "user", "content": f"Headline: {headline}"},
    ]
def score_headline(headline, llm_fn=None):
    messages = build_sentiment_prompt(headline)
    if llm_fn is not None:
        response = llm_fn(messages)
    else:
        import ollama
        response = ollama.chat(model="llama3.2", messages=messages)["message"]["content"]
    return parse_score(response)
def score_headlines(headlines, llm_fn=None):
    return [score_headline(h, llm_fn) for h in headlines]
def aggregate_sentiment(scores):
    if not scores:
        return 0.0
    return max(-1.0, min(1.0, sum(scores) / len(scores)))

def sentiment_to_signal(sentiment, threshold=0.1):
    return 1 if sentiment > threshold else 0
class SentimentSignal:
    def __init__(self, llm_fn=None, threshold=0.1):
        self._llm_fn = llm_fn; self._threshold = threshold; self._history = []
    def score(self, headline):
        s = score_headline(headline, self._llm_fn)
        self._history.append({"headline": headline, "score": s})
        return s
    def score_many(self, headlines):
        return [self.score(h) for h in headlines]
    def signal_from(self, headlines):
        return sentiment_to_signal(aggregate_sentiment(self.score_many(headlines)), self._threshold)
    def history(self):
        return list(self._history)
    def clear_history(self):
        self._history.clear()
import math
def _synthetic(n=20):
    prices = [100.0*(1+0.3*math.sin(i*2*math.pi/20)) for i in range(n)]
    dates  = __import__("pandas").date_range("2023-01-01", periods=n, freq="B")
    close  = __import__("pandas").Series(prices, index=dates)
    return __import__("pandas").DataFrame({
        "Open": close.shift(1).fillna(close.iloc[0]),
        "High": close*1.01, "Low": close*0.99,
        "Close": close,
        "Volume": __import__("pandas").Series([1_000_000]*n, index=dates),
    })

def _compute_returns(df): return df["Close"].pct_change()
def _compute_equity(r):   return (1+r.fillna(0)).cumprod()
def _max_dd(eq):
    peak = eq.cummax(); return float(((eq-peak)/peak).min())
def _sharpe(r):
    c = r.dropna()
    if len(c)==0 or c.std()==0: return 0.0
    return float(c.mean()/c.std()*(252**0.5))
def run_backtest(df, signals, label=""):
    mr  = _compute_returns(df)
    pos = signals.shift(1).fillna(0)
    sr  = pos * mr; eq = _compute_equity(sr)
    c   = sr.dropna(); n = len(c); tr = float(eq.iloc[-1]-1.0)
    base = 1.0+tr
    ar  = float(base**(252.0/max(n,1))-1) if base>0 else -1.0
    pd = __import__("pandas")
    return {
        "label": label, "total_return": tr, "annualized_return": ar,
        "sharpe_ratio": _sharpe(sr), "max_drawdown": _max_dd(eq),
        "win_rate": float((c>0).sum()/max(n,1)),
        "n_trades": int((pos.diff().fillna(0)!=0).sum()),
        "equity": eq,
    }


In [ ]:
import pandas as pd

SCENARIOS = [BULLISH_HEADLINES[:2], NEUTRAL_HEADLINES[:2], BEARISH_HEADLINES[:2]]
df = _synthetic(n=20)
ss = SentimentSignal(llm_fn=_mock_llm, threshold=0.1)
raw_signals = {}
for i, date in enumerate(df.index):
    raw_signals[date] = ss.signal_from(SCENARIOS[i % 3])

signal_series = pd.Series(raw_signals, index=df.index)
result = run_backtest(df, signal_series, "Sentiment")
bah    = run_backtest(df, pd.Series(1, index=df.index), "Buy-Hold")

# Assertions
assert len(ss.history()) == len(df.index) * 2,     f"expected {len(df.index)*2} history entries, got {len(ss.history())}"
assert set(signal_series.unique()).issubset({0, 1})
assert not signal_series.isna().any()
assert abs(result["equity"].iloc[-1] - (1 + result["total_return"])) < 1e-9
assert result["max_drawdown"] <= 1e-9
assert result["n_trades"] >= 0

# Bullish-only signals
ss2 = SentimentSignal(llm_fn=_mock_llm)
bull_sig = ss2.signal_from(BULLISH_HEADLINES)
assert bull_sig == 1, f"bullish → expected 1, got {bull_sig}"
bear_sig = ss2.signal_from(BEARISH_HEADLINES)
assert bear_sig == 0, f"bearish → expected 0, got {bear_sig}"

for label, r in [("Sentiment", result), ("Buy-Hold", bah)]:
    print(f"\n── {label} ─────────────────────")
    print(f"  Total return:    {r['total_return']:.2%}")
    print(f"  Sharpe ratio:    {r['sharpe_ratio']:.3f}")
    print(f"  Max drawdown:    {r['max_drawdown']:.2%}")
    print(f"  Trades:          {r['n_trades']}")
print("\nSolution smoke-test passed.")
